In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
BEST_PARAMETERS_MODEL_DIR = ROOT / "models"
BASELINE_MODEL_DIR = ROOT / "models" / "baseline"
OUTPUT_DIR = ROOT / "results" / "model_evaluation"

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features

TEST_CONFIGS = {
    "cicids2017": ["cicids2017", "iot23", "unsw_nb15"],
    "unsw_nb15": ["unsw_nb15", "cicids2017", "iot23"],
    "iot23": ["iot23", "cicids2017", "unsw_nb15"],
    "cicids2017_unsw_nb15": ["iot23"],
    "cicids2017_iot23": ["unsw_nb15"],
    "unsw_nb15_iot23": ["cicids2017"],
}

TEST_DATASETS = [
    "cicids2017",
    "iot23",
    "unsw_nb15",
]

MODELS_TO_EVALUATE = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

In [2]:
columns_to_load = feature_columns + ["label_binary"]
test_dataframes = {}

for dataset_name in TEST_DATASETS:
    test_dataframes[dataset_name] = pd.read_parquet(
        DATA_DIR / f"{dataset_name}_test.parquet",
        columns=columns_to_load,
    )

for training_name, test_names in TEST_CONFIGS.items():
    for model_name in MODELS_TO_EVALUATE:
        baseline_path = (BASELINE_MODEL_DIR / model_name / f"{training_name}.joblib")
        best_parameters_path = (BEST_PARAMETERS_MODEL_DIR / model_name / f"{training_name}.joblib")

        baseline_pipeline = joblib.load(baseline_path)
        best_parameters_pipeline = joblib.load(best_parameters_path)

        model_versions = [
            ("baseline", "padrão", baseline_pipeline),
            ("best_parameters", "melhores parâmetros", best_parameters_pipeline),
        ]

        report_dir = OUTPUT_DIR / training_name / model_name
        report_dir.mkdir(parents=True, exist_ok=True)

        for version, version_label, pipeline in model_versions:
            for test_name in test_names:
                test_df = test_dataframes[test_name]
                X_test = test_df[feature_columns]
                y_test = test_df["label_binary"].eq("ATTACK").astype("int8")

                predictions = pipeline.predict(X_test)

                report = classification_report(
                    y_test,
                    predictions,
                    labels=[0, 1],
                    target_names=["BENIGN", "ATTACK"],
                    digits=4,
                    zero_division=0,
                )

                matrix = pd.DataFrame(
                    confusion_matrix(y_test, predictions, labels=[0, 1]),
                    index=["Real BENIGN", "Real ATTACK"],
                    columns=["Predito BENIGN", "Predito ATTACK"],
                )

                title = (
                    f"Treino: {training_name} | Modelo: {model_name} | "
                    f"Configuração: {version_label} | Teste: {test_name}"
                )

                report_path = report_dir / f"{version}_{test_name}.txt"
                matrix_path = (
                    report_dir / f"{version}_{test_name}_confusion_matrix.txt"
                )

                report_path.write_text(
                    f"{title}\n\n{report}",
                    encoding="utf-8",
                )
                matrix_path.write_text(
                    f"{title}\n\n{matrix}",
                    encoding="utf-8",
                )

In [7]:
def show_reports(
    training_name,
    test_name,
    show_baseline=True,
    show_with_best_parameters=False,
    show_classification_report=True,
    show_confusion_matrix=True,
):
    versions = []

    if show_baseline:
        versions.append("baseline")
    if show_with_best_parameters:
        versions.append("best_parameters")

    for model_name in MODELS_TO_EVALUATE:
        report_dir = OUTPUT_DIR / training_name / model_name

        for version in versions:
            if show_classification_report:
                report_path = report_dir / f"{version}_{test_name}.txt"
                print(report_path.read_text(encoding="utf-8"))

            if show_confusion_matrix:
                matrix_path = (
                    report_dir / f"{version}_{test_name}_confusion_matrix.txt"
                )
                print(matrix_path.read_text(encoding="utf-8"))

In [8]:
show_reports("cicids2017", "cicids2017")

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9972    0.9953    0.9963     80740
      ATTACK     0.9815    0.9889    0.9852     20185

    accuracy                         0.9941    100925
   macro avg     0.9894    0.9921    0.9907    100925
weighted avg     0.9941    0.9941    0.9941    100925

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: cicids2017

             Predito BENIGN  Predito ATTACK
Real BENIGN           80364             376
Real ATTACK             224           19961
Treino: cicids2017 | Modelo: random_forest | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9983    0.9952    0.9968     80740
      ATTACK     0.9811    0.9934    0.9872     20185

    accuracy                         0.9948    100925
   macro avg     0.9897    0.9943    0.9920    100925
weighted

In [9]:
show_reports("cicids2017", "unsw_nb15")

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.7957    0.9708    0.8746     80740
      ATTACK     0.0244    0.0029    0.0052     20185

    accuracy                         0.7772    100925
   macro avg     0.4101    0.4869    0.4399    100925
weighted avg     0.6414    0.7772    0.7007    100925

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           78382            2358
Real ATTACK           20126              59
Treino: cicids2017 | Modelo: random_forest | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8000    0.9998    0.8888     80740
      ATTACK     0.0000    0.0000    0.0000     20185

    accuracy                         0.7999    100925
   macro avg     0.4000    0.4999    0.4444    100925
weighted av

In [6]:
show_reports("cicids2017_iot23", "unsw_nb15")

Treino: cicids2017_iot23 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8101    0.9934    0.8925     80740
      ATTACK     0.7231    0.0686    0.1253     20185

    accuracy                         0.8085    100925
   macro avg     0.7666    0.5310    0.5089    100925
weighted avg     0.7927    0.8085    0.7390    100925

Treino: cicids2017_iot23 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           80210             530
Real ATTACK           18801            1384
Treino: cicids2017_iot23 | Modelo: decision_tree | Configuração: melhores parâmetros | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8101    0.9934    0.8925     80740
      ATTACK     0.7231    0.0686    0.1253     20185

    accuracy                         0.8085    100925
   macro avg     0.7666    0.5310 